In [0]:
# ============================================================
# FASE 2A — Transform Silver: limpieza y enriquecimiento
# ============================================================
from pyspark.sql.functions import (
    col, trim, split, explode,
    collect_list, concat_ws, to_date,
    round as spark_round
)

# ----------------------------------------------------------
# 1. Cargar datos desde Raw (sin llamadas a API)
# ----------------------------------------------------------
df_movies  = spark.table("etl_cine.peliculas_italianas_raw")
df_generos = spark.table("etl_cine.generos_raw")

print(f"Registros en raw: {df_movies.count()}")
print(f"Géneros disponibles: {df_generos.count()}")

# ----------------------------------------------------------
# 2. Limpieza básica
# ----------------------------------------------------------
df_limpio = df_movies \
    .filter(col("anio_estreno").isNotNull()) \
    .filter(col("titulo").isNotNull()) \
    .filter(col("num_votos") >= 10) \
    .withColumn("titulo",           trim(col("titulo"))) \
    .withColumn("titulo_original",  trim(col("titulo_original"))) \
    .withColumn("popularidad",      spark_round(col("popularidad"), 2)) \
    .withColumn("valoracion_media", spark_round(col("valoracion_media"), 2)) \
    .withColumn("fecha_estreno",    to_date(col("fecha_estreno"), "yyyy-MM-dd"))

print(f"Registros tras limpieza: {df_limpio.count()}")

# ----------------------------------------------------------
# 3. Resolver géneros: IDs → nombres reales
#    genero_ids "37,18" → explode → join → reagrupar
# ----------------------------------------------------------
df_exploded = df_limpio \
    .withColumn("genero_id", explode(split(col("genero_ids"), ","))) \
    .withColumn("genero_id", trim(col("genero_id")))

df_silver = df_exploded \
    .join(df_generos, on="genero_id", how="left") \
    .groupBy(
        "pelicula_id", "titulo", "titulo_original", "fecha_estreno",
        "anio_estreno", "popularidad", "valoracion_media",
        "num_votos", "idioma_original", "descripcion"
    ) \
    .agg(concat_ws(", ", collect_list("genero_nombre")).alias("generos"))

print(f"Registros en silver: {df_silver.count()}")
df_silver.show(5, truncate=False)

# ----------------------------------------------------------
# 4. Guardar capa Silver
# ----------------------------------------------------------
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("etl_cine.peliculas_italianas_silver")

print("✓ Tabla silver guardada: etl_cine.peliculas_italianas_silver")